# Exploración de Datos — CNN+LSTM Biomecánica
**Proyecto:** Reconocimiento de Ejercicios  
**Equipo:** Christopher Zúñiga (C28730) · Adrian Arrieta Orozco (B70734)

In [6]:
import cv2, mediapipe as mp, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

In [7]:
# Define the video and label
video_path = "dataset_limpio/deadlift/deadlift_01.mp4"
video_name = "deadlift_01.mp4"
label = "deadlift"

In [8]:
ap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print("Error: Could not open video file.")
else:
    print("Video file opened successfully!")
# Get video properties
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
print(f"Total frames: {frame_count}, FPS: {fps}")
 
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode
options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='pose_landmarker_full.task'),
    running_mode=VisionRunningMode.IMAGE)
# List to collect all our frame data
dataset_rows = []
with PoseLandmarker.create_from_options(options) as landmarker:
    frame_index = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Convert frame to MediaPipe Image object
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        # Apply detect() to each frame
        pose_landmarker_result = landmarker.detect(mp_image)
        
        # Check if any landmarks were detected
        if pose_landmarker_result.pose_landmarks:
            # We assume there's one person per frame, so we take index [0]
            landmarks = pose_landmarker_result.pose_landmarks[0]
            
            # Basic metadata for the frame
            row_data = {
                'video_name': video_name,
                'label': label,
                'frame_index': frame_index
            }
            
            # MediaPipe Pose has 33 landmarks. We'll flatten them into the row.
            for i, landmark in enumerate(landmarks):
                row_data[f'x{i}'] = landmark.x
                row_data[f'y{i}'] = landmark.y
                row_data[f'z{i}'] = landmark.z
                row_data[f'vis{i}'] = landmark.visibility
                
            dataset_rows.append(row_data)
            
        frame_index += 1
        
    cap.release()
# Convert the collected data into a DataFrame
df = pd.DataFrame(dataset_rows)
# Save to a CSV dataset file
output_csv = "pose_landmarks_dataset.csv"
df.to_csv(output_csv, index=False)
print(f"Extraction complete! Saved {len(df)} frames to '{output_csv}'.")
# Preview the first few rows of the generated dataset
df.head()

Video file opened successfully!
Total frames: 30, FPS: 30.0


I0000 00:00:1777260420.354615 3242255 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1
W0000 00:00:1777260420.484954 3268264 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777260420.491882 3268266 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Extraction complete! Saved 29 frames to 'pose_landmarks_dataset.csv'.


,video_name,label,frame_index,x0,y0,z0,vis0,x1,y1,z1,...,z30,vis30,x31,y31,z31,vis31,x32,y32,z32,vis32
0,deadlift_01.mp4,deadlift,0,0.683238,0.477310,-0.723228,0.999967,0.690567,0.457754,-0.706991,...,-0.099390,0.757113,0.734471,0.971903,-0.275696,0.925351,0.633281,0.971648,-0.271014,0.939336
1,deadlift_01.mp4,deadlift,1,0.683499,0.476626,-0.710239,0.999971,0.691103,0.456390,-0.694131,...,-0.101834,0.764345,0.734195,0.972947,-0.284138,0.933919,0.634339,0.972488,-0.271937,0.944296
2,deadlift_01.mp4,deadlift,2,0.682195,0.452999,-0.702665,0.999981,0.689915,0.433839,-0.685782,...,-0.116103,0.845333,0.736953,0.977325,-0.311870,0.954869,0.633161,0.973726,-0.290698,0.964059
3,deadlift_01.mp4,deadlift,3,0.682466,0.441791,-0.694467,0.999992,0.690352,0.423060,-0.677881,...,-0.087057,0.847425,0.734845,0.975905,-0.274285,0.963935,0.631908,0.972644,-0.255422,0.971145
4,deadlift_01.mp4,deadlift,4,0.680852,0.432573,-0.753762,0.999988,0.689922,0.412954,-0.736986,...,-0.078800,0.855522,0.739789,0.978703,-0.280482,0.970783,0.633429,0.974580,-0.262157,0.975595


In [9]:
# Step 3 — Missing data analysis

In [10]:
# Step 4 — Histograms: landmark coordinate distributions

In [11]:
# Step 5 — Heatmaps: landmark correlations

In [12]:
# Step 6 — Joint angle distributions per exercise